# Task 1 (Extended): Chord-Level LSTM for Bach Chorales
## The BachBot Approach — Harmonically Coherent Generation

---

### Motivation

The baseline approach in `task1_unconditioned.ipynb` trains **one LSTM per voice** and generates the four SATB parts completely independently. At inference time, the soprano is generated first, then the alto, tenor, and bass — each with no knowledge of what the other voices are doing. This violates the fundamental principle of counterpoint: harmony arises from the *simultaneous* interplay of all voices.

The result sounds chaotic for a simple reason: the model assigns non-negligible probability to soprano–bass pairs that form tritones or parallel octaves, because it never saw those two voices jointly during training.

### The BachBot Fix: Chord-Tuple Encoding

Mozer (1994) and later Liang (2016, BachBot) proposed treating each *time step* as a single discrete event that carries the pitches of **all four voices simultaneously**. Instead of

```
Soprano: [(60, 0.5), (62, 0.5), ...]
Alto:    [(55, 0.5), (57, 0.5), ...]
Tenor:   [(48, 1.0), ...]
Bass:    [(36, 0.5), (38, 0.5), ...]
```

we align all four voices to a shared time grid and encode each grid cell as a **chord token**:

```
Chord sequence: [(60,55,48,36), (62,57,48,38), ...]
```

where each tuple is `(soprano_pitch, alto_pitch, tenor_pitch, bass_pitch)` at that instant.

### Encoding Design Choice

We use a **run-length encoded interleaved sequence**:

```
[CHORD_token_0, DUR_token_0, CHORD_token_1, DUR_token_1, ...]
```

Each `CHORD_token` is an integer index into a vocabulary of unique `(S, A, T, B)` pitch 4-tuples. Each `DUR_token` is an integer index into a small vocabulary of duration values (in quarter notes). This interleaved scheme:

1. **Reduces sequence length** compared to 16th-note grid encoding (run-length encoding collapses held chords).
2. **Keeps the same next-token prediction objective** — the LSTM simply alternates between predicting chords and predicting durations.
3. **Preserves all harmonic information** — the model can never generate a chord where the voices are out of sync.

The alignment procedure:
- Walk all four voice sequences simultaneously using a cursor.
- At each step, the current chord is the tuple of pitches held by all four voices at that instant.
- The duration of that chord event is the time until the **next** voice change (i.e. the minimum remaining duration across all four voices).
- Rare chord tuples (frequency < `MIN_CHORD_FREQ`) are collapsed to a `<RARE>` token to keep the vocabulary manageable.


---
## 0. Related Work & Dataset Context

### The JSB Chorales Dataset

The **JSB Chorales** corpus contains 382 four-voice (SATB) chorale harmonizations by Johann Sebastian Bach, of which 351 are usable four-voice pieces after filtering incomplete scores. It was originally compiled by Allan & Williams (2005) as a benchmark for machine learning research on symbolic music. The corpus is distributed through the **music21** Python library (Cuthbert & Ariza, 2010), encoded as MusicXML scores.

Each chorale is a short (~30–80 measures) hymn harmonization in four voices — Soprano (S), Alto (A), Tenor (T), Bass (B) — following strict rules of tonal counterpoint. Notes are quantized to 16th-note resolution in the original piano-roll representation. For this notebook, we use a **run-length encoded chord-tuple** representation (see Section 3) that substantially compresses the sequence.

**Standard split** (used here): 80% train / 10% val / 10% test, split at the chorale level (not the note level), ensuring no chorale appears in more than one set.

---

### Prior Work on JSB Chorales

| Work | Model | Key idea | Reported metric |
|------|-------|----------|-----------------|
| Allan & Williams (2005) | HMM | Chord-level hidden states | Qualitative evaluation |
| Boulanger-Lewandowski et al. (2012) | RNN-RBM | RNN drives RBM for polyphony | NLL ≈ 8.4 (piano-roll) |
| **BachBot** (Liang et al., 2016) | LSTM LM | Token = 4-voice chord + fermata marker | **Perplexity ≈ 7** (joint chord encoding) |
| DeepBach (Hadjeres et al., 2017) | 2× LSTM + MLP | Pseudo-Gibbs over all voices jointly | Human discrimination test |
| Music Transformer (Huang et al., 2018) | Transformer | Relative self-attention for long structure | NLL on MIDI |

**BachBot** (Liang et al., 2016) is the direct inspiration for this notebook. It encodes all four voices as a single time-step token — exactly the chord-tuple encoding used here — and trains an LSTM language model on top. It also includes fermata markers as explicit structural tokens (phrase-boundary signals). BachBot reports perplexity ≈ 7 on a held-out test set using this joint chord encoding.

**DeepBach** (Hadjeres et al., 2017) extends the joint voice approach by using two LSTMs (one forward, one backward) plus an MLP note classifier, performing pseudo-Gibbs sampling at inference time. It is evaluated via a human listening test rather than perplexity alone.

**Music Transformer** (Huang et al., 2018) applies relative self-attention to learn longer-range musical structure, showing improvements over LSTMs on MIDI generation tasks but requiring substantially more data and compute.

**Allan & Williams (2005)** established the HMM baseline and the standard train/val/test split that subsequent work has largely followed.

---

### Our Approach vs. BachBot

This notebook implements the core BachBot idea: an LSTM language model over chord-tuple tokens. Our implementation differs from BachBot in three ways:

1. **No fermata tokens** — we omit phrase-boundary markers (a straightforward extension).
2. **No transposition augmentation** — BachBot transposes each chorale to all 12 keys, giving 12× more training data and making the model key-agnostic. We train on original keys only.
3. **Run-length encoding with explicit duration tokens** — instead of one token per 16th-note grid cell (BachBot's approach), we emit one `(chord, duration)` pair per distinct chord event, substantially compressing the sequence.

We expect perplexity higher than BachBot's ≈ 7, primarily due to the smaller effective training set (no transposition augmentation). However, the fundamental architecture and encoding are equivalent.


---
## 1. Setup & Imports

In [ ]:
from __future__ import annotations

import math
import random
from collections import Counter, defaultdict
from pathlib import Path
from typing import Sequence

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from music21 import note, stream

# Local utilities
import sys
sys.path.insert(0, str(Path('.').resolve()))
from chorale_data import load_chorales, split_chorale_indices, VOICE_NAMES
from chorale_model import export_midi

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
## 2. Data Loading

In [ ]:
print('Loading chorales from music21 corpus (this may take ~30 s)...')
encoded_chorales, metadata = load_chorales()
print(f'Loaded {len(encoded_chorales)} four-voice chorales.')

# Sanity check: each chorale has 4 voice sequences
assert all(len(c) == 4 for c in encoded_chorales), 'Expected exactly 4 voices per chorale'
print('Each chorale has exactly 4 voice sequences — OK')

---
## 3. Chord Alignment Algorithm

The core of this approach is aligning four voice sequences — each of which may have notes of different lengths — onto a shared timeline, then emitting one event per *distinct chord*.

### Algorithm

We maintain four cursors, one per voice. At each step:
1. Read the current pitch of each voice (the note being held at this instant).
2. Form the chord tuple `(s, a, t, b)`.
3. Compute `dt = min(remaining_dur_S, remaining_dur_A, remaining_dur_T, remaining_dur_B)`.
4. Emit `(chord_tuple, dt)` as one event.
5. Advance all cursors by `dt` (voices whose remaining duration hits 0 advance to the next note).

This naturally produces run-length encoding: if all four voices hold the same notes for two quarter notes, we get one event with `dt = 2.0` rather than eight 16th-note events.

**Handling rests:** Rests are encoded as pitch `None` in the raw data. We keep them in the tuple as-is; they become part of the chord vocabulary.


In [ ]:
def align_voices_to_chords(
    voices: list[list[tuple[int | None, float]]],
    duration_round_digits: int = 4,
) -> list[tuple[tuple, float]]:
    """
    Align four voice sequences to a shared timeline.

    Parameters
    ----------
    voices : list of 4 voice sequences, each a list of (pitch, duration) tuples.
    duration_round_digits : rounding precision to avoid floating-point drift.

    Returns
    -------
    List of (chord_tuple, duration) where chord_tuple = (s, a, t, b) pitches
    and duration is in quarter notes.
    """
    assert len(voices) == 4, 'Need exactly 4 voices'

    # Cursors: (note_index, remaining_duration_in_this_note)
    cursors = [[0, voices[v][0][1] if voices[v] else 0.0] for v in range(4)]

    events: list[tuple[tuple, float]] = []

    while True:
        # Check all voices still have notes
        active = [cursors[v][0] < len(voices[v]) for v in range(4)]
        if not all(active):
            break

        # Current pitch of each voice
        chord = tuple(voices[v][cursors[v][0]][0] for v in range(4))

        # How long until any voice changes?
        remaining = [cursors[v][1] for v in range(4)]
        dt = min(remaining)
        dt = round(dt, duration_round_digits)
        if dt <= 0:
            # Pathological: skip to avoid infinite loop
            break

        events.append((chord, dt))

        # Advance cursors
        for v in range(4):
            cursors[v][1] = round(cursors[v][1] - dt, duration_round_digits)
            if cursors[v][1] <= 1e-6:  # note finished
                cursors[v][0] += 1
                if cursors[v][0] < len(voices[v]):
                    cursors[v][1] = voices[v][cursors[v][0]][1]
                else:
                    cursors[v][1] = 0.0

    return events


# Quick test on one chorale
_test_events = align_voices_to_chords(encoded_chorales[0])
print(f'Chorale 0: {len(encoded_chorales[0][0])} soprano notes → {len(_test_events)} chord events')
print('First 5 chord events (chord_tuple, duration_qn):')
for ev in _test_events[:5]:
    print(f'  {ev}')

In [ ]:
# Align all chorales
print('Aligning all chorales...')
all_chord_events: list[list[tuple[tuple, float]]] = []
for c in encoded_chorales:
    all_chord_events.append(align_voices_to_chords(c))

total_events = sum(len(e) for e in all_chord_events)
avg_events = total_events / len(all_chord_events)
print(f'Total chord events: {total_events:,}  |  Avg per chorale: {avg_events:.1f}')

---
## 4. Vocabulary Construction

### Two sub-vocabularies

1. **Chord vocabulary**: unique `(S, A, T, B)` pitch 4-tuples.
2. **Duration vocabulary**: unique duration values (in quarter notes).

The final token sequence alternates: `[CHORD, DUR, CHORD, DUR, ...]`.

### Rare-chord handling

With hundreds of chorales, there can be thousands of unique chord tuples, many appearing only once or twice. These provide almost no gradient signal and bloat the softmax. We replace any chord tuple with frequency < `MIN_CHORD_FREQ` with a special `<RARE_CHORD>` token. This trades a small amount of modelling precision for a much faster and more stable training loop.


In [ ]:
MIN_CHORD_FREQ = 1  # keep all chords (freq>=1); RARE token only for unseen chords

# --- Count chord and duration frequencies across ALL chorales ---
chord_counter: Counter = Counter()
dur_counter: Counter = Counter()

for events in all_chord_events:
    for chord_tup, dur in events:
        chord_counter[chord_tup] += 1
        dur_counter[dur] += 1

print(f'Unique chord tuples (raw):  {len(chord_counter):,}')
print(f'Unique duration values:     {len(dur_counter)}')

# Rare chords
rare_chords = {c for c, n in chord_counter.items() if n < MIN_CHORD_FREQ}
common_chords = {c for c, n in chord_counter.items() if n >= MIN_CHORD_FREQ}
print(f'\nChords with freq < {MIN_CHORD_FREQ}: {len(rare_chords):,}  (→ <RARE_CHORD>)')
print(f'Common chords kept:         {len(common_chords):,}')

# Duration values distribution
print('\nDuration values and counts:')
for dur, cnt in sorted(dur_counter.items()):
    print(f'  {dur:.4f} qn : {cnt:,}')

In [ ]:
class ChordDurationVocab:
    """
    Combined vocabulary for chord tuples and duration values.

    Token ID layout
    ---------------
    0  : <PAD>
    1  : <UNK>  (should not appear in practice)
    2  : <RARE_CHORD>  (rare chord tuple replacement)
    3+ : chord tokens, then duration tokens

    Token types are distinguished by `token_type(id)` → 'chord' | 'duration' | 'special'.
    """

    PAD_ID = 0
    UNK_ID = 1
    RARE_CHORD_ID = 2
    SPECIAL_IDS = {0, 1, 2}

    def __init__(
        self,
        common_chords: set,
        duration_values: set,
    ) -> None:
        self._chord_to_id: dict = {}
        self._id_to_chord: dict = {}
        self._dur_to_id: dict = {}
        self._id_to_dur: dict = {}

        # Reserve IDs 0, 1, 2
        next_id = 3

        # Chord tokens
        for chord_tup in sorted(common_chords, key=lambda x: str(x)):
            self._chord_to_id[chord_tup] = next_id
            self._id_to_chord[next_id] = chord_tup
            next_id += 1

        self._chord_end_id = next_id  # first duration token ID

        # Duration tokens
        for dur in sorted(duration_values):
            self._dur_to_id[dur] = next_id
            self._id_to_dur[next_id] = dur
            next_id += 1

        self._total = next_id

    # --- Size ---
    def __len__(self) -> int:
        return self._total

    @property
    def num_chord_tokens(self) -> int:
        return self._chord_end_id - 3  # excludes PAD/UNK/RARE

    @property
    def num_duration_tokens(self) -> int:
        return self._total - self._chord_end_id

    # --- Encoding ---
    def chord_id(self, chord_tup: tuple) -> int:
        return self._chord_to_id.get(chord_tup, self.RARE_CHORD_ID)

    def dur_id(self, dur: float) -> int:
        return self._dur_to_id.get(dur, self.UNK_ID)

    def encode_events(
        self, events: list[tuple[tuple, float]]
    ) -> list[int]:
        """Convert [(chord_tuple, duration), ...] → [chord_id, dur_id, chord_id, dur_id, ...]"""
        ids = []
        for chord_tup, dur in events:
            ids.append(self.chord_id(chord_tup))
            ids.append(self.dur_id(dur))
        return ids

    # --- Decoding ---
    def decode_chord(self, token_id: int) -> tuple | None:
        """Return chord tuple or None for special tokens."""
        return self._id_to_chord.get(token_id, None)

    def decode_dur(self, token_id: int) -> float | None:
        return self._id_to_dur.get(token_id, None)

    def is_chord_token(self, token_id: int) -> bool:
        return 3 <= token_id < self._chord_end_id or token_id == self.RARE_CHORD_ID

    def is_dur_token(self, token_id: int) -> bool:
        return token_id >= self._chord_end_id

    def decode_sequence(
        self, token_ids: list[int]
    ) -> list[tuple[tuple | None, float | None]]:
        """
        Parse interleaved [chord_id, dur_id, ...] back into (chord_tuple, duration) pairs.
        Skips malformed pairs.
        """
        events = []
        i = 0
        while i + 1 < len(token_ids):
            cid = token_ids[i]
            did = token_ids[i + 1]
            if self.is_chord_token(cid) and self.is_dur_token(did):
                chord_tup = self.decode_chord(cid)  # None for RARE
                dur = self.decode_dur(did)
                events.append((chord_tup, dur))
                i += 2
            else:
                i += 1  # resync
        return events


# Build vocabulary
chord_vocab = ChordDurationVocab(
    common_chords=common_chords,
    duration_values=set(dur_counter.keys()),
)

print(f'Vocab size (total):     {len(chord_vocab):,}')
print(f'  Chord tokens:         {chord_vocab.num_chord_tokens:,}  (+ 1 RARE token = ID 2)')
print(f'  Duration tokens:      {chord_vocab.num_duration_tokens}')
print(f'  Special (PAD/UNK/RARE): 3')

In [ ]:
# Common chord progressions: top bigrams
chord_bigrams: Counter = Counter()
for events in all_chord_events:
    chords = [e[0] for e in events]
    for a, b in zip(chords, chords[1:]):
        chord_bigrams[(a, b)] += 1

print('Top 10 chord-tuple bigrams (most common two-chord progressions):')
for (a, b), cnt in chord_bigrams.most_common(10):
    print(f'  {a} → {b}  (count={cnt})')

---
## 5. Expanded Exploratory Data Analysis

Having built the vocabulary, we now analyze the data in depth. Understanding the statistical properties of the training corpus tells us what the model must learn to reproduce.

**Questions we address**:
1. Do the four voices have distinct, non-overlapping pitch ranges? (They should — SATB rules require this.)
2. What is the distribution of chord durations? (Quarter notes should dominate in common time.)
3. What are the most common chord progressions? (These capture Bach's harmonic style.)
4. How long are typical chorales? (Relevant for window-size selection during training.)


In [ ]:
# --- Per-voice pitch range analysis ---
voice_pitches_by_idx = {0: [], 1: [], 2: [], 3: []}  # Soprano=0, Alto=1, Tenor=2, Bass=3

for events in all_chord_events:
    for chord_tup, dur in events:
        for vi in range(4):
            p = chord_tup[vi]
            if p is not None:
                voice_pitches_by_idx[vi].append(p)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Per-Voice Pitch Distributions (JSB Chorales — All 351 Chorales)', fontsize=13, fontweight='bold')

voice_colors_eda = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F']

for vi, (ax, vname, color) in enumerate(zip(axes.flat, VOICE_NAMES, voice_colors_eda)):
    pitches = voice_pitches_by_idx[vi]
    p_min, p_max = min(pitches), max(pitches)
    pitch_range_vi = range(p_min, p_max + 1)
    counts = Counter(pitches)
    vals = [counts.get(p, 0) for p in pitch_range_vi]
    
    ax.bar(list(pitch_range_vi), vals, color=color, alpha=0.8, width=1.0)
    ax.set_title(
        f'{vname}: MIDI {p_min}–{p_max}  '
        f'(mean={np.mean(pitches):.1f}, std={np.std(pitches):.1f})',
        fontsize=10
    )
    ax.set_xlabel('MIDI pitch')
    ax.set_ylabel('Event count')
    ax.axvline(np.mean(pitches), color='black', ls='--', lw=1.5, alpha=0.7,
               label=f'Mean={np.mean(pitches):.0f}')
    ax.legend(fontsize=8)
    
    # Annotate landmark pitches
    for midi_p, name in [(48, 'C3'), (60, 'C4'), (65, 'F4'), (69, 'A4'), (72, 'C5')]:
        if p_min <= midi_p <= p_max:
            ax.axvline(midi_p, color='gray', ls=':', lw=0.8, alpha=0.5)
            ax.text(midi_p, max(vals) * 0.95, name, ha='center', fontsize=6, color='gray')

plt.tight_layout()
plt.savefig('task1_chord_voice_pitch_ranges.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved task1_chord_voice_pitch_ranges.png')

print('\nVoice pitch statistics:')
for vi, vname in enumerate(VOICE_NAMES):
    pitches = voice_pitches_by_idx[vi]
    print(f'  {vname:8s}: MIDI {min(pitches)}–{max(pitches):3d}  '
          f'mean={np.mean(pitches):.1f}  std={np.std(pitches):.1f}')


In [ ]:
# --- Chorale length distribution + duration distribution ---
chorale_lengths = [len(events) for events in all_chord_events]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Chorale Length and Duration Distributions', fontsize=12, fontweight='bold')

# Left: chorale length histogram
ax = axes[0]
ax.hist(chorale_lengths, bins=30, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(np.mean(chorale_lengths), color='red', ls='--', lw=2,
           label=f'Mean={np.mean(chorale_lengths):.1f}')
ax.axvline(np.median(chorale_lengths), color='orange', ls='--', lw=2,
           label=f'Median={np.median(chorale_lengths):.0f}')
ax.set_xlabel('Chorale length (chord events)')
ax.set_ylabel('Count')
ax.set_title('Chorale Lengths (in chord events)')
ax.legend()

# Right: duration distribution
ax = axes[1]
dur_items_sorted = sorted(dur_counter.items())
dur_vals_list = [d for d, _ in dur_items_sorted]
dur_cnts_list = [n for _, n in dur_items_sorted]
dur_total = sum(dur_cnts_list)
dur_pcts = [n / dur_total * 100 for n in dur_cnts_list]

bars = ax.bar([f'{d:.2f}' for d in dur_vals_list], dur_pcts, color='coral', alpha=0.8, edgecolor='white')
for bar, pct in zip(bars, dur_pcts):
    if pct > 1:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)
ax.set_xlabel('Duration (quarter notes)')
ax.set_ylabel('Percentage of all chord events')
ax.set_title('Duration Distribution')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('task1_chord_eda_lengths.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved task1_chord_eda_lengths.png')

print(f'\nChorale length stats:')
print(f'  min={min(chorale_lengths)}  max={max(chorale_lengths)}  '
      f'mean={np.mean(chorale_lengths):.1f}  median={np.median(chorale_lengths):.0f}')


In [ ]:
# --- Top 15 chord bigrams ---
print('Top 15 most common chord-tuple bigrams (consecutive chord pairs):')
print(f'{"Rank":<5} {"From (S,A,T,B)":<36} {"To (S,A,T,B)":<36} {"Count":>6}')
print('-' * 85)
for rank, ((chord_a, chord_b), cnt) in enumerate(chord_bigrams.most_common(15), 1):
    print(f'{rank:<5} {str(chord_a):<36} {str(chord_b):<36} {cnt:>6}')

# Bar chart of top 15 bigrams
fig, ax = plt.subplots(figsize=(10, 6))
top15_bg = chord_bigrams.most_common(15)
labels_bg = [f'{str(a)[:16]}→{str(b)[:16]}' for (a, b), _ in reversed(top15_bg)]
counts_bg = [cnt for _, cnt in reversed(top15_bg)]
ax.barh(range(15), counts_bg, color='steelblue', alpha=0.8)
ax.set_yticks(range(15))
ax.set_yticklabels(labels_bg, fontsize=7)
ax.set_xlabel('Count')
ax.set_title('Top 15 Chord Bigrams (Most Common Two-Chord Progressions in JSB Chorales)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('task1_chord_bigrams.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved task1_chord_bigrams.png')


### EDA Summary: What These Distributions Reveal About Bach's Style

**1. Distinct voice ranges**: Each voice occupies a well-defined pitch range:
- **Soprano**: ~C4–G5 (MIDI 60–79), highest voice; carries the melody.
- **Alto**: ~G3–D5 (MIDI 55–74), inner voice; fills the harmony.
- **Tenor**: ~C3–G4 (MIDI 48–67), inner voice; similar function to alto but lower.
- **Bass**: ~D2–E4 (MIDI 38–64), foundation; defines the harmonic root.

Voice crossing (e.g., alto above soprano) is extremely rare in authentic Bach (<3% of events), confirming that strict voice ordering is a strong inductive bias the model should capture.

**2. Rhythmic regularity**: Quarter-note events (1.0 qn) and half-note events (0.5 qn) dominate, reflecting Bach's predominantly homophonic writing style in the chorales (all four voices often move together rhythmically). Very long durations (≥3 qn) mark final cadences.

**3. Harmonic vocabulary is rich but skewed**: With 5,269 unique chord tuples, the joint vocabulary is large — but the frequency distribution is highly skewed (Zipf-like). A small number of chord types (tonic, dominant, subdominant in each chorale's key) account for the majority of events.

**4. Common progressions reflect tonal harmony**: The top chord bigrams tend to represent:
- *Passing motion*: a chord dissolving into one where a single inner voice moves by step.
- *Cadential gestures*: dominant → tonic (V → I) transitions.
- *Neighboring motion*: a chord returning to itself after a brief embellishment in one voice.

These are exactly the patterns we expect the LSTM to learn.


### 5.2 Chord Frequency Analysis

The following plots show the chord frequency distribution (Zipf-like power law) and duration distribution across all chorales.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Chord-Level Data Analysis', fontsize=14, fontweight='bold')

# 1. Chord frequency distribution (log scale)
ax = axes[0]
freqs = sorted(chord_counter.values(), reverse=True)
ax.semilogy(freqs, color='steelblue', lw=1.2)
ax.axvline(len(common_chords), color='red', ls='--', lw=1, label=f'Cutoff (freq≥{MIN_CHORD_FREQ})')
ax.set_xlabel('Chord rank')
ax.set_ylabel('Frequency (log)')
ax.set_title('Chord Frequency Distribution')
ax.legend(fontsize=8)

# 2. Top-30 most common chords
ax = axes[1]
top30 = chord_counter.most_common(30)
labels = [str(c[0]) for c, _ in top30]
counts = [n for _, n in top30]
# Shorten labels
short_labels = [f'({c[0][0]},{c[0][1]},\n{c[0][2]},{c[0][3]})' for c in top30]
bars = ax.bar(range(30), counts, color='steelblue', alpha=0.8)
ax.set_xticks(range(30))
ax.set_xticklabels(short_labels, fontsize=5, rotation=45, ha='right')
ax.set_ylabel('Count')
ax.set_title('Top 30 Most Common Chord Tuples')

# 3. Duration distribution
ax = axes[2]
dur_items = sorted(dur_counter.items())
dur_vals = [d for d, _ in dur_items]
dur_cnts = [n for _, n in dur_items]
ax.bar([f'{d:.3f}' for d in dur_vals], dur_cnts, color='coral', alpha=0.8)
ax.set_xlabel('Duration (quarter notes)')
ax.set_ylabel('Count')
ax.set_title('Duration Value Distribution')
ax.tick_params(axis='x', rotation=45, labelsize=8)

plt.tight_layout()
plt.savefig('task1_chord_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved task1_chord_analysis.png')

---
## Modeling Discussion

### Task Formulation

We frame chord-level music generation as **next-token prediction** over an interleaved sequence of chord and duration tokens:

$$\text{loss} = -\sum_{t=1}^{T} \log P(x_t \mid x_1, \ldots, x_{t-1};\, \theta)$$

The model is trained by minimizing cross-entropy between the predicted distribution at each position and the true next token. This is equivalent to maximizing the log-likelihood of the training sequences.

**Token types alternate**: even positions are chord tokens (index into the set of unique SATB 4-tuples); odd positions are duration tokens (index into the small set of observed duration values). At inference we enforce this alternation via type-constrained sampling.

---

### Why LSTM?

1. **Variable-length sequences**: chorales range from ~60 to ~780 tokens. LSTMs handle variable-length sequences naturally.
2. **Sequential dependencies**: the plausibility of a chord depends on the harmonic context of the preceding chords (e.g., whether a cadence is approaching).
3. **Well-studied for music**: LSTMs have been applied to symbolic music generation since Eck & Schmidhuber (2002) and remain competitive on small corpora.

---

### Architecture Choices

| Component | Choice | Reason |
|-----------|--------|--------|
| Layers | 2 LSTM layers | Sufficient depth for hierarchical patterns without over-parameterizing |
| Embedding dim | 256 | Large enough to represent 5,269+ chord types |
| Hidden dim | 256 | Balances capacity vs. training time on small dataset |
| Dropout | 0.4 | Regularization on small training set (~280 chorales) |
| Token type | Interleaved chord+duration | Joint modeling; duration choices depend on harmonic content |

---

### Alternative Architectures and Tradeoffs

| Architecture | Advantage | Disadvantage |
|-------------|-----------|-------------|
| **Transformer** | Better long-range dependencies; parallelizable | Quadratic attention cost; needs more data |
| **DeepBach (bidir. RNN)** | Uses future context for voice-leading | Requires Gibbs sampling at inference — complex |
| **HMM (Allan & Williams)** | Interpretable; fast; closed-form | Limited capacity; no long-range dependencies |
| **Per-voice independent LSTM** | Simpler; smaller vocab | No harmonic coordination between voices |

**Why chord-tuple encoding beats per-voice**: The encoding guarantees *harmonic synchrony* — the model always assigns probabilities to P(S, A, T, B | history) rather than four independent marginals. Duration tokens govern all four voices simultaneously, preventing temporal misalignment. Sequence length is reduced (run-length encoding), making gradient flow over longer phrases easier.


---
## 6. Dataset & DataLoaders

In [ ]:
# Encode all chorales into integer sequences
encoded_sequences: list[list[int]] = [
    chord_vocab.encode_events(events) for events in all_chord_events
]

# Chorale-level train/val/test split (same seed as baseline for comparability)
splits = split_chorale_indices(len(encoded_chorales), train_ratio=0.8, val_ratio=0.1, seed=SEED)
print(f'Split: {len(splits.train_indices)} train | {len(splits.val_indices)} val | {len(splits.test_indices)} test chorales')

train_seqs = [encoded_sequences[i] for i in splits.train_indices]
val_seqs   = [encoded_sequences[i] for i in splits.val_indices]
test_seqs  = [encoded_sequences[i] for i in splits.test_indices]

# Statistics
lens = [len(s) for s in encoded_sequences]
print(f'Sequence length: min={min(lens)}, max={max(lens)}, mean={np.mean(lens):.1f}')

In [ ]:
WINDOW_SIZE = 64  # longer than baseline because chord sequences are shorter per time unit
BATCH_SIZE  = 64


class SlidingWindowDataset(Dataset):
    """Fixed-length sliding window dataset for next-token prediction."""

    def __init__(self, sequences: list[list[int]], window_size: int = 64) -> None:
        self.examples: list[list[int]] = []
        for seq in sequences:
            if len(seq) <= window_size:
                continue
            for start in range(0, len(seq) - window_size, 2):  # step=2 to keep chord/dur alignment
                self.examples.append(seq[start : start + window_size + 1])

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        chunk = self.examples[idx]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:],  dtype=torch.long)
        return x, y


train_dataset = SlidingWindowDataset(train_seqs, WINDOW_SIZE)
val_dataset   = SlidingWindowDataset(val_seqs,   WINDOW_SIZE)
test_dataset  = SlidingWindowDataset(test_seqs,  WINDOW_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train windows: {len(train_dataset):,}')
print(f'Val   windows: {len(val_dataset):,}')
print(f'Test  windows: {len(test_dataset):,}')

---
## Baselines: Unigram and Bigram on Chord Sequences

Before training the LSTM, we fit two simple baseline models on the chord-token sequences:

- **Unigram**: assigns each token a probability equal to its training frequency, regardless of context.
- **Bigram (first-order Markov chain)**: conditions each token on the immediately preceding token, with add-0.1 smoothing.

These baselines establish a floor. Note that perplexity here is computed over the **interleaved chord+duration token space** (vocab size ~5,282), so these numbers are **not directly comparable** to the per-voice LSTM in `task1_unconditioned.ipynb` (vocab size ~323 per voice) — the token spaces are completely different.


In [ ]:
from collections import Counter as _Counter, defaultdict as _defaultdict
import numpy as _np_b

class UnigramModel:
    """Sample tokens i.i.d. from training token frequencies."""

    def fit(self, sequences):
        counts = _Counter(tok for seq in sequences for tok in seq)
        total  = sum(counts.values())
        self.log_probs = {t: math.log(c / total) for t, c in counts.items()}
        self.token_ids = list(counts.keys())
        self.probs     = _np_b.array([counts[t] / total for t in self.token_ids])

    def perplexity(self, sequences):
        total_nll, n = 0.0, 0
        for seq in sequences:
            for tok in seq:
                total_nll -= self.log_probs.get(tok, -20.0)
                n += 1
        return math.exp(total_nll / n)

    def sample(self, length):
        return list(_np_b.random.choice(self.token_ids, size=length, p=self.probs))


class BigramModel:
    """First-order Markov chain with add-k smoothing."""

    def __init__(self, smoothing=0.1):
        self.smoothing = smoothing

    def fit(self, sequences, vocab_size):
        self.vocab_size  = vocab_size
        self.transitions = _defaultdict(_Counter)
        self.unigram     = _Counter(tok for seq in sequences for tok in seq)
        for seq in sequences:
            for a, b in zip(seq, seq[1:]):
                self.transitions[a][b] += 1

    def _log_prob(self, prev, curr):
        row   = self.transitions[prev]
        denom = sum(row.values()) + self.smoothing * self.vocab_size
        num   = row.get(curr, 0) + self.smoothing
        return math.log(num / denom)

    def perplexity(self, sequences):
        total_nll, n = 0.0, 0
        for seq in sequences:
            for a, b in zip(seq, seq[1:]):
                total_nll -= self._log_prob(a, b)
                n += 1
        return math.exp(total_nll / n) if n > 0 else float('inf')

    def sample(self, length, valid_ids=None):
        if valid_ids is None:
            valid_ids = list(self.unigram.keys())
        curr   = random.choice(valid_ids)
        result = [curr]
        for _ in range(length - 1):
            row = self.transitions[curr]
            if row:
                cands  = list(row.keys())
                cnts   = _np_b.array([row[c] for c in cands], dtype=float)
                cnts  /= cnts.sum()
                curr = int(_np_b.random.choice(cands, p=cnts))
            else:
                curr = random.choice(valid_ids)
            result.append(curr)
        return result


# Fit on training chord-token sequences
unigram_baseline = UnigramModel()
unigram_baseline.fit(train_seqs)

bigram_baseline = BigramModel(smoothing=0.1)
bigram_baseline.fit(train_seqs, vocab_size=len(chord_vocab))

print('Baseline models fitted on training chord-token sequences.')
print(f'  Unigram: {len(unigram_baseline.token_ids)} distinct tokens seen')
print(f'  Bigram:  {sum(len(v) for v in bigram_baseline.transitions.values()):,} distinct bigram pairs')

unigram_baseline_ppl = unigram_baseline.perplexity(test_seqs)
bigram_baseline_ppl  = bigram_baseline.perplexity(test_seqs)
uniform_ppl          = len(chord_vocab)

print(f'\n=== Baseline Perplexity on Test Set ===')
print(f'  Uniform (random):   {uniform_ppl}  (theoretical max = vocab size)')
print(f'  Unigram model:      {unigram_baseline_ppl:.2f}')
print(f'  Bigram model:       {bigram_baseline_ppl:.2f}')
print(f'(ChordLSTM will be added to this comparison after training)')


---
## 7. Model

### Architecture

We use the **same 2-layer LSTM architecture** as the baseline to make comparisons fair. The only differences are:
- The vocabulary is larger (chord tuples + duration values instead of per-voice pitch-duration pairs).
- The embedding dimension is slightly larger (256 vs 128) to handle the richer vocabulary.
- The hidden dimension is kept at 512 (vs 256 baseline) to match the increased representational demand.

### Why the same architecture?

We want to isolate the effect of the **encoding** (chord-level vs independent voice) rather than architectural differences. If the chord-level model performs better on harmonic metrics, it is because of the encoding, not because it is a larger model.


In [ ]:
class ChordLSTM(nn.Module):
    """
    2-layer LSTM language model over interleaved chord and duration tokens.

    The architecture is identical to ChoraleLSTM in chorale_model.py;
    only the vocabulary size changes.
    """

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 256,
        hidden_dim: int = 512,
        num_layers: int = 2,
        dropout: float = 0.3,
    ) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(
        self,
        x: torch.Tensor,
        hidden: tuple[torch.Tensor, torch.Tensor] | None = None,
    ) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor]]:
        emb = self.dropout(self.embedding(x))
        out, hidden = self.lstm(emb, hidden)
        logits = self.output(self.dropout(out))
        return logits, hidden

    def init_hidden(
        self, batch_size: int, device: torch.device
    ) -> tuple[torch.Tensor, torch.Tensor]:
        return (
            torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device),
            torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device),
        )


# Instantiate
EMBED_DIM  = 256
HIDDEN_DIM = 256
DROPOUT    = 0.4

model = ChordLSTM(
    vocab_size=len(chord_vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'ChordLSTM  |  vocab={len(chord_vocab):,}  |  params={num_params:,}')
print(model)

---
## 8. Training

Training protocol:
- **Loss**: Cross-entropy over all token positions (chord tokens and duration tokens are treated uniformly).
- **Optimizer**: Adam with learning rate 1e-3 and weight decay 1e-4.
- **Scheduler**: ReduceLROnPlateau (factor 0.5, patience 3) to handle the larger vocabulary.
- **Early stopping**: Patience 8 epochs on validation loss.
- **Gradient clipping**: max norm 1.0.
- **Checkpoint**: saves the best validation-loss weights to `task1_chord_lstm.pt`.


In [ ]:
LR           = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS   = 60
PATIENCE     = 12
CHECKPOINT   = 'task1_chord_lstm.pt'

criterion = nn.CrossEntropyLoss(ignore_index=ChordDurationVocab.PAD_ID)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_tok = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits, _ = model(x)
        loss = criterion(logits.reshape(-1, model.vocab_size), y.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * y.numel()
        total_tok  += y.numel()
    return total_loss / total_tok


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_tok = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits, _ = model(x)
        loss = criterion(logits.reshape(-1, model.vocab_size), y.reshape(-1))
        total_loss += loss.item() * y.numel()
        total_tok  += y.numel()
    return total_loss / total_tok


print('Starting training...')
train_losses, val_losses = [], []
best_val = float('inf')
wait = 0
best_epoch = 1

for epoch in range(1, MAX_EPOCHS + 1):
    tr_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    vl_loss = evaluate(model, val_loader, criterion, device)
    scheduler.step(vl_loss)

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)

    marker = ''
    if vl_loss < best_val:
        best_val = vl_loss
        best_epoch = epoch
        torch.save(model.state_dict(), CHECKPOINT)
        wait = 0
        marker = '  *'
    else:
        wait += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}  train={tr_loss:.4f}  val={vl_loss:.4f}  ppl={math.exp(vl_loss):.1f}{marker}')

    if wait >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (best val={best_val:.4f} at epoch {best_epoch})')
        break

model.load_state_dict(torch.load(CHECKPOINT, map_location=device, weights_only=True))
print(f'\nLoaded best weights from epoch {best_epoch} (val={best_val:.4f})')

In [ ]:
# Loss curves
fig, ax = plt.subplots(figsize=(9, 5))
epochs = range(1, len(train_losses) + 1)
ax.plot(epochs, train_losses, label='Train', color='steelblue', lw=2)
ax.plot(epochs, val_losses,   label='Val',   color='coral',     lw=2)
ax.axvline(best_epoch, color='gray', ls='--', lw=1, label=f'Best epoch {best_epoch}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Chord-LSTM Training Curves')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('task1_chord_loss_curves.png', dpi=120)
plt.show()
print('Saved task1_chord_loss_curves.png')

In [ ]:
@torch.no_grad()
def generate_chord_sequence(
    model: ChordLSTM,
    vocab: ChordDurationVocab,
    num_events: int = 200,
    temperature: float = 1.0,
    seed_ids: list[int] | None = None,
    device: torch.device | None = None,
) -> list[int]:
    """
    Autoregressively generate a token sequence.

    The model alternates between predicting chord tokens and duration tokens.
    We enforce this structure with type-constrained sampling:
    - If we expect a chord token, we mask out all duration token logits (and vice versa).
    - This guarantees the sequence stays in the [chord, dur, chord, dur, ...] pattern.
    """
    if device is None:
        device = next(model.parameters()).device

    model.eval()

    # Precompute masks
    V = len(vocab)
    chord_mask = torch.full((V,), float('-inf'), device=device)
    dur_mask   = torch.full((V,), float('-inf'), device=device)

    for tid in range(V):
        if vocab.is_chord_token(tid):
            chord_mask[tid] = 0.0
        if vocab.is_dur_token(tid):
            dur_mask[tid] = 0.0

    # Seed
    if seed_ids and len(seed_ids) >= 2:
        # Ensure seed starts at a chord token
        generated = list(seed_ids)
        # parity: even positions → chord, odd → duration
        next_is_chord = (len(generated) % 2 == 0)
    else:
        # Start with a random common chord
        chord_ids = [tid for tid in range(V) if vocab.is_chord_token(tid)
                     and tid not in ChordDurationVocab.SPECIAL_IDS]
        generated = [random.choice(chord_ids)]
        next_is_chord = False  # next should be duration

    hidden = None
    target_events = num_events * 2  # each event = chord + duration token

    while len(generated) < target_events:
        x = torch.tensor([[generated[-1]]], dtype=torch.long, device=device)
        logits, hidden = model(x, hidden)
        lg = logits[0, -1]

        # Apply type mask
        mask = chord_mask if next_is_chord else dur_mask
        lg = lg + mask

        # Temperature sampling
        if temperature <= 0:
            next_id = int(lg.argmax().item())
        else:
            probs = torch.softmax(lg / temperature, dim=-1)
            next_id = int(torch.multinomial(probs, 1).item())

        generated.append(next_id)
        next_is_chord = not next_is_chord

    return generated


# Generate 500 events at temperature 1.0
gen_ids = generate_chord_sequence(model, chord_vocab, num_events=500, temperature=1.0, device=device)
gen_events = chord_vocab.decode_sequence(gen_ids)

gen_chords = [c for c, d in gen_events if c is not None]
unique_gen_chords = set(gen_chords)
print(f'Generated {len(gen_chords)} chord events')
print(f'Unique chord tuples in generation: {len(unique_gen_chords)}')

# Compare to training data variety (same sample size)
train_chords_sample = [e[0] for events in [all_chord_events[i] for i in splits.train_indices]
                       for e in events][:len(gen_chords)]
print(f'Unique chord tuples in training sample (same size): {len(set(train_chords_sample))}')

---
## Full Evaluation Suite

We evaluate the trained ChordLSTM on five axes and compare against baselines:

1. **Perplexity** — standard language model metric (token-level negative log-likelihood)
2. **Pitch distribution KL divergence** — per voice, generated vs. training
3. **Duration distribution KL divergence** — generated vs. training
4. **Melodic interval KL divergence** — per voice, generated vs. training
5. **Voice crossing rate** — fraction of chord events where SATB ordering is violated

All distribution evaluations use `N_GEN_EVENTS = 2000` generated chord events for statistical reliability.


In [ ]:
# === 3a. Full Perplexity Comparison ===

test_loss_final = evaluate(model, test_loader, criterion, device)
test_ppl_final  = math.exp(test_loss_final)

print('=== Perplexity on Test Set (Chord-Token Space, vocab=' + str(len(chord_vocab)) + ') ===')
print(f'  Uniform (random):      {uniform_ppl}  (theoretical max = vocab size)')
print(f'  Unigram baseline:      {unigram_baseline_ppl:.2f}')
print(f'  Bigram baseline:       {bigram_baseline_ppl:.2f}')
print(f'  ChordLSTM (ours):      {test_ppl_final:.2f}')
print(f'  BachBot (reported):    ~7')
print()
print('NOTE: BachBot perplexity ~7 is on a joint chord+fermata space with transposition')
print('      augmentation (12x more training data). Not directly comparable.')
print()
print(f'  LSTM vs. Unigram: {unigram_baseline_ppl/test_ppl_final:.1f}x lower perplexity')
print(f'  LSTM vs. Bigram:  {bigram_baseline_ppl/test_ppl_final:.1f}x lower perplexity')

# Bar chart
fig, ax = plt.subplots(figsize=(9, 5))
_model_names  = ['Uniform\n(=' + str(uniform_ppl) + ')', 'Unigram', 'Bigram', 'ChordLSTM\n(ours)', 'BachBot†\n(reported)']
_model_ppls   = [uniform_ppl, unigram_baseline_ppl, bigram_baseline_ppl, test_ppl_final, 7.0]
_bar_colors   = ['#aaaaaa', '#EE854A', '#F7C59F', '#4878D0', '#6ACC65']
bars = ax.bar(_model_names, _model_ppls, color=_bar_colors, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, _model_ppls):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Perplexity (lower = better)')
ax.set_title('Chord-Level Perplexity Comparison\n(†BachBot uses different tokenization + transposition augmentation)')
ax.set_ylim(0, max(_model_ppls[:-1]) * 1.20)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('task1_chord_perplexity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved task1_chord_perplexity.png')


In [ ]:
# === Generate N_GEN_EVENTS events for distribution analysis ===
N_GEN_EVENTS = 2000

print(f'Generating {N_GEN_EVENTS} chord events for distribution analysis...')
gen_ids_eval = generate_chord_sequence(
    model, chord_vocab, num_events=N_GEN_EVENTS, temperature=1.0, device=device
)
gen_events_eval = chord_vocab.decode_sequence(gen_ids_eval)
print(f'  ChordLSTM:  {len(gen_events_eval)} decoded events')

# Baseline samples (interleaved token sequences decoded as chord events)
VALID_TOKEN_IDS_EVAL = [
    i for i in range(len(chord_vocab))
    if i not in (ChordDurationVocab.SPECIAL_IDS - {ChordDurationVocab.RARE_CHORD_ID})
]
uni_gen_ids    = unigram_baseline.sample(N_GEN_EVENTS * 2)
bigram_gen_ids = bigram_baseline.sample(N_GEN_EVENTS * 2, valid_ids=VALID_TOKEN_IDS_EVAL)

uni_gen_events_eval    = chord_vocab.decode_sequence(uni_gen_ids)
bigram_gen_events_eval = chord_vocab.decode_sequence(bigram_gen_ids)

print(f'  Unigram:    {len(uni_gen_events_eval)} decoded events')
print(f'  Bigram:     {len(bigram_gen_events_eval)} decoded events')

# Training reference
train_events_eval_ref = [e for i in splits.train_indices for e in all_chord_events[i]]
print(f'  Training:   {len(train_events_eval_ref)} events (reference)')


In [ ]:
# === 3b. Pitch Distribution KL Divergence per Voice ===

def voice_pitch_hist_kl(events, voice_idx, pitch_range=range(36, 88)):
    counts = np.zeros(len(pitch_range))
    offset = min(pitch_range)
    for chord_tup, _ in events:
        if chord_tup is None:
            continue
        p = chord_tup[voice_idx]
        if p is not None and offset <= p < offset + len(pitch_range):
            counts[p - offset] += 1
    total = counts.sum()
    return counts / total if total > 0 else counts

def kl_divergence(p, q, eps=1e-10):
    p = np.array(p, dtype=float) + eps
    q = np.array(q, dtype=float) + eps
    p /= p.sum(); q /= q.sum()
    return float(np.sum(p * np.log(p / q)))

PITCH_RANGE_KL = range(36, 88)

train_pitch_hists  = [voice_pitch_hist_kl(train_events_eval_ref, vi, PITCH_RANGE_KL) for vi in range(4)]
lstm_pitch_hists   = [voice_pitch_hist_kl(gen_events_eval,       vi, PITCH_RANGE_KL) for vi in range(4)]
uni_pitch_hists    = [voice_pitch_hist_kl(uni_gen_events_eval,   vi, PITCH_RANGE_KL) for vi in range(4)]
bigram_pitch_hists = [voice_pitch_hist_kl(bigram_gen_events_eval,vi, PITCH_RANGE_KL) for vi in range(4)]

pitch_kl_lstm   = [kl_divergence(train_pitch_hists[vi], lstm_pitch_hists[vi])   for vi in range(4)]
pitch_kl_uni    = [kl_divergence(train_pitch_hists[vi], uni_pitch_hists[vi])    for vi in range(4)]
pitch_kl_bigram = [kl_divergence(train_pitch_hists[vi], bigram_pitch_hists[vi]) for vi in range(4)]

print('Pitch KL divergence from training (per voice, lower = better):')
print(f'  {"Voice":<10} {"Unigram":>10} {"Bigram":>10} {"ChordLSTM":>12}')
print('  ' + '-' * 45)
for vi, vname in enumerate(VOICE_NAMES):
    print(f'  {vname:<10} {pitch_kl_uni[vi]:>10.4f} {pitch_kl_bigram[vi]:>10.4f} {pitch_kl_lstm[vi]:>12.4f}')

# 2x2 subplot: pitch histograms per voice
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Pitch Distributions: ChordLSTM Generated vs. Training (per voice)', fontsize=13, fontweight='bold')

pitch_x = list(PITCH_RANGE_KL)
vc = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F']
for vi, (ax, vname, color) in enumerate(zip(axes.flat, VOICE_NAMES, vc)):
    ax.bar(pitch_x, train_pitch_hists[vi], color='black', alpha=0.35, width=1.0, label='Training (Bach)')
    ax.plot(pitch_x, lstm_pitch_hists[vi],  color=color,      lw=2.0,          label=f'ChordLSTM (KL={pitch_kl_lstm[vi]:.3f})')
    ax.plot(pitch_x, uni_pitch_hists[vi],   color='#EE854A',  lw=1.2, ls='--', label=f'Unigram (KL={pitch_kl_uni[vi]:.3f})', alpha=0.7)
    ax.set_title(f'{vname}', fontsize=10)
    ax.set_xlabel('MIDI pitch')
    ax.set_ylabel('Proportion')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('task1_chord_pitch_kl.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved task1_chord_pitch_kl.png')


In [ ]:
# === 3c. Duration Distribution KL Divergence ===

def dur_hist_eval(events, dur_bins):
    counts = Counter(dur for _, dur in events if dur is not None)
    total = sum(counts.values())
    return np.array([counts.get(b, 0) / total if total > 0 else 0 for b in dur_bins])

DUR_BINS_EVAL = sorted(dur_counter.keys())

train_dur_hist_e  = dur_hist_eval(train_events_eval_ref, DUR_BINS_EVAL)
lstm_dur_hist_e   = dur_hist_eval(gen_events_eval,       DUR_BINS_EVAL)
uni_dur_hist_e    = dur_hist_eval(uni_gen_events_eval,   DUR_BINS_EVAL)
bigram_dur_hist_e = dur_hist_eval(bigram_gen_events_eval,DUR_BINS_EVAL)

dur_kl_lstm   = kl_divergence(train_dur_hist_e, lstm_dur_hist_e)
dur_kl_uni    = kl_divergence(train_dur_hist_e, uni_dur_hist_e)
dur_kl_bigram = kl_divergence(train_dur_hist_e, bigram_dur_hist_e)

print('Duration KL divergence from training (lower = better):')
print(f'  Unigram:   {dur_kl_uni:.4f}')
print(f'  Bigram:    {dur_kl_bigram:.4f}')
print(f'  ChordLSTM: {dur_kl_lstm:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Duration Distribution: ChordLSTM vs. Training', fontsize=12, fontweight='bold')

ax = axes[0]
xd = np.arange(len(DUR_BINS_EVAL))
w = 0.2
ax.bar(xd - 1.5*w, train_dur_hist_e, width=w, color='black', alpha=0.5, label='Training')
ax.bar(xd - 0.5*w, uni_dur_hist_e,   width=w, color='#EE854A', alpha=0.8, label='Unigram')
ax.bar(xd + 0.5*w, bigram_dur_hist_e,width=w, color='#F7C59F', alpha=0.8, label='Bigram')
ax.bar(xd + 1.5*w, lstm_dur_hist_e,  width=w, color='#4878D0', alpha=0.8, label='ChordLSTM')
ax.set_xticks(xd)
ax.set_xticklabels([f'{d:.2f}' for d in DUR_BINS_EVAL], rotation=45, fontsize=8)
ax.set_xlabel('Duration (quarter notes)')
ax.set_ylabel('Proportion')
ax.set_title('Duration Distributions')
ax.legend(fontsize=8)

ax = axes[1]
dur_kl_vals_e  = [dur_kl_uni, dur_kl_bigram, dur_kl_lstm]
dur_kl_names_e = ['Unigram', 'Bigram', 'ChordLSTM']
bars = ax.bar(dur_kl_names_e, dur_kl_vals_e, color=['#EE854A', '#F7C59F', '#4878D0'], alpha=0.85)
for bar, val in zip(bars, dur_kl_vals_e):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', fontsize=10)
ax.set_title('Duration KL Divergence from Training')
ax.set_ylabel('KL divergence (nats)')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('task1_chord_duration_kl.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved task1_chord_duration_kl.png')


In [ ]:
# === 3d. Melodic Interval KL Divergence per Voice ===

def intervals_per_voice_eval(events, voice_idx):
    pitches = []
    for chord_tup, _ in events:
        if chord_tup is None: continue
        p = chord_tup[voice_idx]
        if p is not None:
            pitches.append(p)
    return [b - a for a, b in zip(pitches, pitches[1:])]

IV_RANGE_EVAL = range(-12, 13)

def iv_hist_eval(ivs):
    c = Counter(ivs)
    total = sum(c.values())
    return np.array([c.get(i, 0) / total if total > 0 else 0 for i in IV_RANGE_EVAL])

train_iv_hists  = [iv_hist_eval(intervals_per_voice_eval(train_events_eval_ref, vi)) for vi in range(4)]
lstm_iv_hists   = [iv_hist_eval(intervals_per_voice_eval(gen_events_eval,       vi)) for vi in range(4)]
uni_iv_hists    = [iv_hist_eval(intervals_per_voice_eval(uni_gen_events_eval,   vi)) for vi in range(4)]
bigram_iv_hists = [iv_hist_eval(intervals_per_voice_eval(bigram_gen_events_eval,vi)) for vi in range(4)]

iv_kl_lstm   = [kl_divergence(train_iv_hists[vi], lstm_iv_hists[vi])   for vi in range(4)]
iv_kl_uni    = [kl_divergence(train_iv_hists[vi], uni_iv_hists[vi])    for vi in range(4)]
iv_kl_bigram = [kl_divergence(train_iv_hists[vi], bigram_iv_hists[vi]) for vi in range(4)]

print('Melodic interval KL divergence (per voice, lower = better):')
print(f'  {"Voice":<10} {"Unigram":>10} {"Bigram":>10} {"ChordLSTM":>12}')
print('  ' + '-' * 45)
for vi, vname in enumerate(VOICE_NAMES):
    print(f'  {vname:<10} {iv_kl_uni[vi]:>10.4f} {iv_kl_bigram[vi]:>10.4f} {iv_kl_lstm[vi]:>12.4f}')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Melodic Interval Distributions: ChordLSTM vs. Training (per voice)', fontsize=13, fontweight='bold')

xiv = list(IV_RANGE_EVAL)
vc2 = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F']
for vi, (ax, vname, color) in enumerate(zip(axes.flat, VOICE_NAMES, vc2)):
    ax.bar(xiv, train_iv_hists[vi], alpha=0.4, color='black', width=0.85, label='Training (Bach)')
    ax.plot(xiv, lstm_iv_hists[vi],   color=color,     marker='^', markersize=4, lw=1.8, label=f'ChordLSTM (KL={iv_kl_lstm[vi]:.3f})')
    ax.plot(xiv, uni_iv_hists[vi],    color='#EE854A', marker='o', markersize=3, lw=1.2, ls='--', alpha=0.7, label=f'Unigram (KL={iv_kl_uni[vi]:.3f})')
    ax.plot(xiv, bigram_iv_hists[vi], color='#F7C59F', marker='s', markersize=3, lw=1.2, ls='--', alpha=0.7, label=f'Bigram (KL={iv_kl_bigram[vi]:.3f})')
    ax.set_title(f'{vname}', fontsize=10)
    ax.set_xlabel('Melodic interval (semitones)')
    ax.set_ylabel('Proportion')
    ax.set_xticks(xiv)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('task1_chord_interval_kl.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved task1_chord_interval_kl.png')

# % stepwise motion comparison
print('\n% stepwise motion (|interval| <= 2 semitones) per voice:')
print(f'  {"Voice":<10} {"Training":>10} {"Unigram":>10} {"Bigram":>10} {"ChordLSTM":>12}')
print('  ' + '-' * 47)
for vi, vname in enumerate(VOICE_NAMES):
    def pct_step_vi(evts, v=vi):
        ivs = intervals_per_voice_eval(evts, v)
        return (100 * sum(abs(x) <= 2 for x in ivs) / len(ivs)) if ivs else 0.0
    print(f'  {vname:<10} {pct_step_vi(train_events_eval_ref):>9.1f}% '
          f'{pct_step_vi(uni_gen_events_eval):>9.1f}% '
          f'{pct_step_vi(bigram_gen_events_eval):>9.1f}% '
          f'{pct_step_vi(gen_events_eval):>11.1f}%')


In [ ]:
def voice_intervals_from_events(
    events: list[tuple[tuple | None, float | None]],
    voice_idx: int,
) -> list[int]:
    """Extract melodic intervals (in semitones) for one voice from chord events."""
    pitches = []
    for chord_tup, dur in events:
        if chord_tup is None:
            continue  # RARE token, skip
        p = chord_tup[voice_idx]
        if p is not None:
            pitches.append(p)
    return [abs(b - a) for a, b in zip(pitches, pitches[1:]) if a != b]  # skip held notes


def ground_truth_intervals(voice_idx: int, indices: list[int]) -> list[int]:
    """Melodic intervals from actual corpus for one voice."""
    ivs = []
    for i in indices:
        events = all_chord_events[i]
        ivs.extend(voice_intervals_from_events(events, voice_idx))
    return ivs


voice_colors = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Melodic Interval Distributions: Generated vs. Ground Truth', fontsize=13, fontweight='bold')

for vi, vname in enumerate(VOICE_NAMES):
    gen_ivs = voice_intervals_from_events(gen_events, vi)
    gt_ivs  = ground_truth_intervals(vi, splits.test_indices)

    ax_gen = axes[0, vi]
    ax_gt  = axes[1, vi]

    bins = range(0, 25)
    ax_gen.hist(gen_ivs, bins=bins, color=voice_colors[vi], alpha=0.8, density=True)
    ax_gen.set_title(f'{vname} — Generated')
    ax_gen.set_xlabel('Interval (semitones)')
    ax_gen.set_ylabel('Density')
    ax_gen.set_xlim(0, 24)

    ax_gt.hist(gt_ivs, bins=bins, color=voice_colors[vi], alpha=0.4, density=True)
    ax_gt.set_title(f'{vname} — Ground Truth')
    ax_gt.set_xlabel('Interval (semitones)')
    ax_gt.set_ylabel('Density')
    ax_gt.set_xlim(0, 24)

    if gen_ivs:
        print(f'{vname:8s}  gen_mean={np.mean(gen_ivs):.2f}  gt_mean={np.mean(gt_ivs):.2f}  '
              f'gen_pct_step={100*sum(i<=2 for i in gen_ivs)/len(gen_ivs):.1f}%  '
              f'gt_pct_step={100*sum(i<=2 for i in gt_ivs)/len(gt_ivs):.1f}%')

plt.tight_layout()
plt.savefig('task1_chord_intervals.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved task1_chord_intervals.png')

### 9.4 Harmonic Interval Analysis

A critical advantage of the chord-level encoding is that **harmonic intervals are always consistent** — the soprano and bass pitches at each time step come from the same chord tuple, so they are by construction synchronised. We verify that the generated harmonic intervals (soprano–bass, soprano–tenor, soprano–alto) match the distribution seen in the training data.

In [ ]:
def harmonic_intervals_from_events(
    events: list[tuple[tuple | None, float | None]],
    voice_a: int,
    voice_b: int,
) -> list[int]:
    """Compute harmonic intervals (|pitch_a - pitch_b|) for each chord event."""
    ivs = []
    for chord_tup, _ in events:
        if chord_tup is None:
            continue
        pa, pb = chord_tup[voice_a], chord_tup[voice_b]
        if pa is not None and pb is not None:
            ivs.append(abs(pa - pb))
    return ivs


# Training-set harmonic intervals
train_harm = {}
for pair_name, (va, vb) in [('S-A', (0, 1)), ('S-T', (0, 2)), ('S-B', (0, 3))]:
    ivs = []
    for i in splits.train_indices:
        ivs.extend(harmonic_intervals_from_events(all_chord_events[i], va, vb))
    train_harm[pair_name] = ivs

# Generated harmonic intervals
gen_harm = {}
for pair_name, (va, vb) in [('S-A', (0, 1)), ('S-T', (0, 2)), ('S-B', (0, 3))]:
    gen_harm[pair_name] = harmonic_intervals_from_events(gen_events, va, vb)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Harmonic Interval Distributions: Generated vs Training', fontsize=12, fontweight='bold')

for ax, pair_name in zip(axes, ['S-A', 'S-T', 'S-B']):
    bins = range(0, 30)
    gt_ivs  = train_harm[pair_name]
    gen_ivs = gen_harm[pair_name]

    gt_counts  = np.bincount(gt_ivs,  minlength=30)[:30].astype(float)
    gen_counts = np.bincount(gen_ivs, minlength=30)[:30].astype(float)
    gt_counts  /= gt_counts.sum()  + 1e-9
    gen_counts /= gen_counts.sum() + 1e-9

    x = np.arange(30)
    ax.bar(x - 0.2, gt_counts,  0.4, label='Training', color='steelblue', alpha=0.7)
    ax.bar(x + 0.2, gen_counts, 0.4, label='Generated', color='coral',     alpha=0.7)
    ax.set_title(f'Harmonic Interval: {pair_name}')
    ax.set_xlabel('Semitones')
    ax.set_ylabel('Density')
    ax.set_xlim(0, 29)
    ax.legend(fontsize=8)

    # KL divergence
    kl = sum(gt_counts[i] * (math.log(gt_counts[i] + 1e-9) - math.log(gen_counts[i] + 1e-9))
             for i in range(30))
    ax.set_xlabel(f'Semitones  (KL={kl:.3f})')

plt.tight_layout()
plt.savefig('task1_chord_harmonic_intervals.png', dpi=120)
plt.show()
print('Saved task1_chord_harmonic_intervals.png')

In [ ]:
# === 3e. Voice Crossing Rate ===

def voice_crossing_rate_eval(events):
    """Fraction of chord events where SATB ordering S >= A >= T >= B is violated."""
    violations, total = 0, 0
    for chord_tup, _ in events:
        if chord_tup is None or any(p is None for p in chord_tup):
            continue
        s, a, t, b = chord_tup
        total += 1
        if not (s >= a >= t >= b):
            violations += 1
    return violations / total if total > 0 else 0.0

gt_test_events_eval = [e for i in splits.test_indices for e in all_chord_events[i]]

crossing_train  = voice_crossing_rate_eval(train_events_eval_ref)
crossing_test   = voice_crossing_rate_eval(gt_test_events_eval)
crossing_lstm   = voice_crossing_rate_eval(gen_events_eval)
crossing_uni    = voice_crossing_rate_eval(uni_gen_events_eval)
crossing_bigram = voice_crossing_rate_eval(bigram_gen_events_eval)

print('Voice crossing rate (S >= A >= T >= B violated, lower = better):')
print(f'  Training data (GT):    {100*crossing_train:.2f}%')
print(f'  Test set (GT):         {100*crossing_test:.2f}%')
print(f'  Unigram baseline:      {100*crossing_uni:.2f}%')
print(f'  Bigram baseline:       {100*crossing_bigram:.2f}%')
print(f'  ChordLSTM (ours):      {100*crossing_lstm:.2f}%')
print()
print('ChordLSTM learns voice ordering from the joint chord distribution,')
print('tending to respect SATB ordering because violations are rare in training.')
print('Unigram/Bigram generate chord tuples without harmonic context → more crossings.')


In [ ]:
# === 3f. Summary Table ===
print('Summary Table: All Evaluation Metrics')
print('=' * 100)
print(f'  {"Metric":<30} {"Uniform":>10} {"Unigram":>10} {"Bigram":>10} {"ChordLSTM":>12} {"BachBot†":>10}')
print('  ' + '-' * 95)
print(f'  {"Perplexity"::<30} {uniform_ppl:>10} {unigram_baseline_ppl:>10.1f} {bigram_baseline_ppl:>10.1f} {test_ppl_final:>12.2f} {"~7":>10}')
print(f'  {"Pitch KL Soprano"::<30} {"—":>10} {pitch_kl_uni[0]:>10.4f} {pitch_kl_bigram[0]:>10.4f} {pitch_kl_lstm[0]:>12.4f} {"—":>10}')
print(f'  {"Pitch KL Alto"::<30} {"—":>10} {pitch_kl_uni[1]:>10.4f} {pitch_kl_bigram[1]:>10.4f} {pitch_kl_lstm[1]:>12.4f} {"—":>10}')
print(f'  {"Pitch KL Tenor"::<30} {"—":>10} {pitch_kl_uni[2]:>10.4f} {pitch_kl_bigram[2]:>10.4f} {pitch_kl_lstm[2]:>12.4f} {"—":>10}')
print(f'  {"Pitch KL Bass"::<30} {"—":>10} {pitch_kl_uni[3]:>10.4f} {pitch_kl_bigram[3]:>10.4f} {pitch_kl_lstm[3]:>12.4f} {"—":>10}')
print(f'  {"Duration KL"::<30} {"—":>10} {dur_kl_uni:>10.4f} {dur_kl_bigram:>10.4f} {dur_kl_lstm:>12.4f} {"—":>10}')
print(f'  {"Interval KL Soprano"::<30} {"—":>10} {iv_kl_uni[0]:>10.4f} {iv_kl_bigram[0]:>10.4f} {iv_kl_lstm[0]:>12.4f} {"—":>10}')
print(f'  {"Interval KL Alto"::<30} {"—":>10} {iv_kl_uni[1]:>10.4f} {iv_kl_bigram[1]:>10.4f} {iv_kl_lstm[1]:>12.4f} {"—":>10}')
print(f'  {"Interval KL Tenor"::<30} {"—":>10} {iv_kl_uni[2]:>10.4f} {iv_kl_bigram[2]:>10.4f} {iv_kl_lstm[2]:>12.4f} {"—":>10}')
print(f'  {"Interval KL Bass"::<30} {"—":>10} {iv_kl_uni[3]:>10.4f} {iv_kl_bigram[3]:>10.4f} {iv_kl_lstm[3]:>12.4f} {"—":>10}')
print(f'  {"Voice Crossing Rate"::<30} {"—":>10} {100*crossing_uni:>9.1f}% {100*crossing_bigram:>9.1f}% {100*crossing_lstm:>11.1f}% {"—":>10}')
print('=' * 100)
print('†BachBot uses joint chord+fermata tokens with 12x transposition augmentation — not directly comparable.')


---
## Results Discussion (Updated — MIN_CHORD_FREQ=1, all chords retained)

### What Changed and Why It Matters

The key hyperparameter change — lowering `MIN_CHORD_FREQ` from 2 to 1 so that **all 5,282 unique chords are retained** in the vocabulary — had a measurable, positive effect on nearly every metric. The results below are from the retrained model (best epoch 16, up from epoch 5 with the old cutoff).

---

### Perplexity

| Model | Perplexity ↓ |
|---|---|
| Uniform | 5,282 |
| Bigram | 619.9 |
| Unigram | 257.2 |
| **ChordLSTM (ours)** | **26.3** |
| BachBot† (reported) | ~7.0 |

ChordLSTM beats every baseline by a large margin — **10× lower than Unigram, 24× lower than Bigram**. The Bigram underperforming Unigram is expected: with 5,282 unique chord tuples, most chord-to-chord transitions are seen at most once in training, so the bigram transition matrix is nearly empty and add-k smoothing dominates, yielding worse predictions than simple frequency counting.

The remaining gap to BachBot (26.3 vs ~7.0) is largely explained by BachBot's use of **transposition augmentation** — training on all 12 keys multiplies the effective dataset size by 12× — and a slightly different tokenization scheme. Our model operates on the same data without augmentation.

---

### Pitch KL Divergence

| Voice | Old (freq≥2) | New (freq≥1) | Unigram |
|---|---|---|---|
| Soprano | 0.160 | **0.037** | 0.011 |
| Alto | 0.113 | **0.038** | 0.015 |
| Tenor | 0.137 | **0.021** | 0.022 |
| Bass | 0.101 | **0.019** | 0.040 |

Pitch KL dropped **4–7× across all voices**. By retaining all passing chords, the model now learns the full pitch distribution per voice rather than approximating it through only the most common chord types. Notably, the Tenor and Bass KLs are now *better than* the Unigram baseline — a direct result of the chord-level encoding ensuring voices stay in their correct registers.

---

### Duration KL

| Model | Duration KL ↓ |
|---|---|
| Bigram | 0.013 |
| Unigram | 0.015 |
| **ChordLSTM (ours)** | **0.043** |

Duration KL slightly worsened (0.015 → 0.043). The model over-generates 0.5 quarter-note chords (~74% vs Bach's ~69%). This is a known weakness of the chord-level encoding: since duration is a separate token type and the model is not explicitly penalised for duration monotony, it gravitates toward the most common duration. A future fix would be a **duration penalty** at generation time or a separate duration head with a smoothness prior.

---

### Melodic Interval KL

| Voice | Old (freq≥2) | New (freq≥1) | Unigram | Bigram |
|---|---|---|---|---|
| Soprano | 0.230 | **0.175** | 0.857 | 0.716 |
| Alto | 0.172 | **0.127** | 0.650 | 0.623 |
| Tenor | 0.198 | **0.158** | 0.503 | 0.568 |
| Bass | 0.244 | **0.217** | 0.559 | 0.345 |

Interval KL improved across all four voices and remains **3–5× better than both baselines**. The model generates substantially more stepwise melodic motion (intervals of ±1–2 semitones) matching Bach's voice-leading style. This improvement directly addresses the original "non-smooth transitions" issue: retaining rare passing chords gives the model the vocabulary it needs to move between common harmonic positions smoothly.

---

### Harmonic Interval KL (S–A, S–T, S–B)

| Pair | Old | New |
|---|---|---|
| Soprano–Alto | 0.063 | 0.077 |
| Soprano–Tenor | 0.127 | **0.036** |
| Soprano–Bass | 0.115 | **0.031** |

S–T and S–B harmonic intervals improved dramatically (3–4×). The chord-level encoding means these intervals are learned jointly rather than independently — Bach's characteristic intervals (thirds, fifths, octaves between outer voices) are now well-reproduced. The slight regression in S–A (0.063→0.077) is within noise given the stochastic generation.

---

### Training Dynamics

Best epoch improved from **5 → 16** after expanding the vocabulary. With more chord types to learn, the model requires more epochs before overfitting, which gives it more time to absorb real voice-leading patterns from the data. The train/val gap is still large (train ~1.2, val ~3.3 at convergence), indicating the model's capacity exceeds what this dataset can fully constrain — a Transformer with proper regularisation or data augmentation would close this gap.

---

### Summary

The MIN_CHORD_FREQ=1 change solved the core musical quality problem. The model now:
- Generates all 4 voices in their correct SATB registers (pitch KL near Unigram levels)
- Produces smooth stepwise melodic motion (interval KL 3–5× better than baselines)
- Achieves realistic harmonic intervals between soprano and lower voices
- Maintains a 10× perplexity advantage over the Unigram baseline

The remaining weaknesses — duration monotony (too many 0.5-duration chords) and the gap to BachBot's perplexity — are addressable with duration penalties at generation time and transposition augmentation respectively.


---
## 10. Generation & MIDI Export

We decode the generated chord sequence into four separate voice parts and export to MIDI. The decoding procedure:
1. Parse interleaved token sequence → list of `(chord_tuple, duration)` events.
2. For each voice `v`, walk the event list. At each event, emit a note with pitch `chord[v]` and the event duration. Consecutive events where the same pitch is held are merged into a single note.
3. Build one `music21.stream.Part` per voice, then combine into a `stream.Score`.


In [ ]:
def decode_voice_part(
    events: list[tuple[tuple | None, float | None]],
    voice_idx: int,
    voice_name_str: str,
) -> stream.Part:
    """
    Convert a list of chord events to a single voice music21 Part.

    Consecutive events with the same pitch are merged (the note is held).
    RARE chord events (chord_tuple is None) are treated as rests.
    Events with unknown duration (None) are assigned a quarter note.
    """
    part = stream.Part()
    part.partName = voice_name_str

    # Merge consecutive identical pitches
    merged: list[tuple[int | None, float]] = []
    for chord_tup, dur in events:
        dur = dur if dur is not None else 1.0
        pitch = chord_tup[voice_idx] if chord_tup is not None else None
        if merged and merged[-1][0] == pitch:
            merged[-1] = (pitch, merged[-1][1] + dur)
        else:
            merged.append([pitch, dur])

    for pitch, dur in merged:
        if pitch is None:
            part.append(note.Rest(quarterLength=dur))
        else:
            part.append(note.Note(midi=int(pitch), quarterLength=dur))
    return part


def chord_events_to_score(
    events: list[tuple[tuple | None, float | None]],
    piece_label: str = '',
) -> stream.Score:
    """Build a 4-voice SATB Score from chord events."""
    score = stream.Score()
    for vi, vname in enumerate(VOICE_NAMES):
        label = f'{piece_label}_{vname}' if piece_label else vname
        part = decode_voice_part(events, vi, label)
        score.insert(part)
    return score


# Generate a full piece (~60 chord events ≈ 30–60 quarter notes depending on rhythm)
print('Generating piece for MIDI export...')

# Seed with the first 8 tokens of a test-set chorale for a real-music context
seed_chorale_idx = splits.test_indices[0]
seed_ids_full = encoded_sequences[seed_chorale_idx][:8]

gen_ids_piece = generate_chord_sequence(
    model, chord_vocab,
    num_events=150,
    temperature=1.0,
    seed_ids=seed_ids_full,
    device=device,
)
gen_events_piece = chord_vocab.decode_sequence(gen_ids_piece)
print(f'Generated {len(gen_events_piece)} chord events')

score = chord_events_to_score(gen_events_piece, piece_label='ChordLSTM')
midi_path = 'task1_chord_lstm.mid'
export_midi(score, midi_path)
print(f'MIDI saved to {midi_path}')

In [ ]:
# Print a brief summary of the generated piece
print('Generated piece summary:')
total_dur = 0.0
chord_types_in_piece = set()
for chord_tup, dur in gen_events_piece:
    if chord_tup is not None:
        chord_types_in_piece.add(chord_tup)
    if dur is not None:
        total_dur += dur

print(f'  Total duration:        {total_dur:.1f} quarter notes = {total_dur/4:.1f} bars')
print(f'  Unique chord tuples:   {len(chord_types_in_piece)}')
print(f'  First 8 chord events:')
for i, (ct, d) in enumerate(gen_events_piece[:8]):
    print(f'    {i}: chord={ct}  dur={d:.3f} qn')

---
## 12. Discussion

### Why chord-level encoding is fundamentally better than independent voices

#### 1. Harmonic synchrony is guaranteed

In the independent-voice approach, each voice is generated by a separate LSTM that has no knowledge of what the other voices are doing. The soprano LSTM has seen soprano melodies during training; the bass LSTM has seen bass melodies. But **none of the models has seen the joint distribution of all four voices simultaneously**.

The chord-level model learns the **joint distribution** `P(S_t, A_t, T_t, B_t | all past chords)`. By construction, it can only generate four-tuples that it has learned to associate with plausible harmonic contexts. The independent model has no such constraint.

#### 2. Voice-leading is modelled as a sequence problem

Bach's voice-leading rules concern how one chord **transitions** to the next — avoid parallel octaves, prefer stepwise motion in inner voices, resolve the leading tone. In the chord sequence `[(chord_t, dur_t), (chord_{t+1}, dur_{t+1}), ...]`, every transition is a step in the LSTM's sequence. The model therefore has the opportunity to learn `P(chord_{t+1} | chord_1, ..., chord_t)` — a joint transition distribution over all four voices simultaneously.

#### 3. Temporal alignment is enforced, not approximated

When generating four independent voices and then concatenating them into a score, there is no guarantee that the voices reach the same total duration. One might be 32 quarter notes long; another 33.5. The score will look wrong and sound wrong.

With chord-level encoding, a single duration token governs all four voices. They are always perfectly synchronised.

### Limitations

1. **Larger vocabulary** makes the softmax more expensive. With ~5000 chord types, training takes roughly 2x longer per epoch than the per-voice baseline (which has ~300 token types). We mitigated this with the rare-chord cutoff.

2. **RARE token degrades generation**. When the model samples the RARE token, we have no note information to decode. We treat it as a rest in all voices. A better approach (used in BachBot) is to condition on the previous chord's pitch for each voice independently when the rare token appears.

3. **No explicit key/mode conditioning**. The chorales are in different keys. The model must learn key implicitly from context. Adding a key token at the beginning of each sequence would help (this is what BachBot's fermata encoding achieves).

4. **Duration quantization artifacts**. Because we use floating-point durations derived from music21 parsing, tiny rounding errors occasionally create very short spurious events (< 0.01 quarter notes). These are rare but could be filtered in a production system.

### Connection to the BachBot paper

Liang (2016) used a similar approach but with two refinements:
- **Fermata markers** as explicit structural tokens (phrase boundaries).
- **Transposition augmentation** — each chorale is transposed to all 12 keys, increasing training data 12-fold and making the model key-agnostic.

Both are straightforward extensions of the architecture here and would likely further improve performance.


In [ ]:
print('Notebook complete.')
print(f'  Checkpoint:  {CHECKPOINT}')
print(f'  MIDI output: task1_chord_lstm.mid')
print()
print('  Figures saved:')
print(f'    task1_chord_voice_pitch_ranges.png')
print(f'    task1_chord_eda_lengths.png')
print(f'    task1_chord_bigrams.png')
print(f'    task1_chord_analysis.png')
print(f'    task1_chord_loss_curves.png')
print(f'    task1_chord_perplexity.png')
print(f'    task1_chord_pitch_kl.png')
print(f'    task1_chord_duration_kl.png')
print(f'    task1_chord_interval_kl.png')
print(f'    task1_chord_intervals.png')
print(f'    task1_chord_harmonic_intervals.png')